Imports & Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
PALETTE = {
    'bg':        '#0D0F1A',
    'card':      '#131625',
    'accent1':   '#6C63FF',   # violet
    'accent2':   '#FF6584',   # coral
    'accent3':   '#43E97B',   # mint
    'accent4':   '#FFD166',   # amber
    'text':      '#E8E8F0',
    'subtext':   '#7C7CA0',
    'churn':     '#FF6584',
    'retain':    '#43E97B',
    'grid':      '#1E2035',
}

In [ ]:
plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['card'],
    'axes.edgecolor':    PALETTE['grid'],
    'axes.labelcolor':   PALETTE['text'],
    'axes.titlecolor':   PALETTE['text'],
    'xtick.color':       PALETTE['subtext'],
    'ytick.color':       PALETTE['subtext'],
    'grid.color':        PALETTE['grid'],
    'grid.linewidth':    0.6,
    'text.color':        PALETTE['text'],
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    14,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'legend.facecolor':  PALETTE['card'],
    'legend.edgecolor':  PALETTE['grid'],
    'legend.labelcolor': PALETTE['text'],
    'figure.dpi':        130,
})

In [ ]:
print("Libraries loaded | Style configured")

Load Dataset (CSV)


In [ ]:
file_path = 'synthetic_churn_data.csv'   # make sure file is in same folder

df = pd.read_csv(file_path)

# Convert signup_date to datetime
df['signup_date'] = pd.to_datetime(df['signup_date'])

# Quick sanity check
print(f"Dataset loaded successfully | Shape: {df.shape}")
print(f"Churn Rate: {df['churn'].mean()*100:.2f}%")

df.head()

Exploratory Data Analysis (EDA)


In [ ]:
print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nChurn distribution:")
print(df['churn'].value_counts())
print(f"\nChurn Rate: {df['churn'].mean()*100:.2f}%")
print("\nNumerical summary:")
df.describe().round(2)

VISUAL 1: Business Overview Dashboard


In [ ]:
fig = plt.figure(figsize=(18, 10), facecolor=PALETTE['bg'])
fig.suptitle('CUSTOMER CHURN — BUSINESS OVERVIEW',
             fontsize=22, fontweight='bold', color=PALETTE['text'],
             y=0.98)
 
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)
 
churn_counts = df['churn'].value_counts()
total        = len(df)
churned      = churn_counts.get(1, 0)
retained     = churn_counts.get(0, 0)
 
# ── KPI Cards (top row text tiles) ───────────────────────────
kpi_ax = fig.add_subplot(gs[0, 0])
kpi_ax.set_facecolor(PALETTE['bg'])
kpi_ax.axis('off')
 
kpis = [
    (f"{total:,}",          "Total Customers",  PALETTE['accent1']),
    (f"{churned:,}",        "Churned",          PALETTE['churn']),
    (f"{retained:,}",       "Retained",         PALETTE['retain']),
    (f"{churned/total*100:.1f}%", "Churn Rate", PALETTE['accent4']),
]
 
for i, (val, label, color) in enumerate(kpis):
    y = 0.88 - i * 0.22
    kpi_ax.text(0.05, y, val,   fontsize=22, fontweight='bold',
                color=color, transform=kpi_ax.transAxes)
    kpi_ax.text(0.05, y-0.08, label, fontsize=9,
                color=PALETTE['subtext'], transform=kpi_ax.transAxes)
kpi_ax.set_title('KEY METRICS', fontsize=11, color=PALETTE['subtext'],
                 loc='left', pad=8)
 
# ── Donut chart ───────────────────────────────────────────────
ax_donut = fig.add_subplot(gs[0, 1])
wedge_colors = [PALETTE['retain'], PALETTE['churn']] 
wedges, texts, autotexts = ax_donut.pie(
    [retained, churned],
    labels=['Retained', 'Churned'],
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.52, edgecolor=PALETTE['bg'], linewidth=3),
)
for at in autotexts:
    at.set_color(PALETTE['bg']); at.set_fontweight('bold'); at.set_fontsize(11)
for t in texts:
    t.set_color(PALETTE['text']); t.set_fontsize(10)
ax_donut.set_title('CHURN vs RETENTION', pad=12)
 
# ── Churn by Plan ─────────────────────────────────────────────
ax_plan = fig.add_subplot(gs[0, 2])
plan_churn = df.groupby('plan_type')['churn'].mean() * 100
bars = ax_plan.bar(plan_churn.index, plan_churn.values,
                   color=[PALETTE['accent1'], PALETTE['accent2'], PALETTE['accent4']],
                   edgecolor=PALETTE['bg'], linewidth=1.5, width=0.55,
                   zorder=3)
ax_plan.yaxis.grid(True, zorder=0)
ax_plan.set_title('CHURN RATE BY PLAN', pad=12)
ax_plan.set_ylabel('Churn Rate (%)')
for bar, val in zip(bars, plan_churn.values):
    ax_plan.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom',
                 fontsize=10, fontweight='bold', color=PALETTE['text'])
 
# ── Churn by Contract ────────────────────────────────────────
ax_contract = fig.add_subplot(gs[1, 0])
contract_churn = df.groupby('contract_type')['churn'].mean() * 100
bars2 = ax_contract.bar(contract_churn.index, contract_churn.values,
                        color=[PALETTE['accent3'], PALETTE['accent2']],
                        edgecolor=PALETTE['bg'], linewidth=1.5, width=0.45,
                        zorder=3)
ax_contract.yaxis.grid(True, zorder=0)
ax_contract.set_title('CHURN BY CONTRACT TYPE', pad=12)
ax_contract.set_ylabel('Churn Rate (%)')
for bar, val in zip(bars2, contract_churn.values):
    ax_contract.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom',
                     fontsize=11, fontweight='bold', color=PALETTE['text'])
 
# ── Churn by Country ─────────────────────────────────────────
ax_country = fig.add_subplot(gs[1, 1:])
country_churn = (df.groupby('country')['churn'].mean() * 100).sort_values(ascending=True)
colors_country = plt.cm.cool(np.linspace(0.2, 0.9, len(country_churn)))
hbars = ax_country.barh(country_churn.index, country_churn.values,
                         color=colors_country, edgecolor=PALETTE['bg'],
                         linewidth=1.2, zorder=3, height=0.6)
ax_country.xaxis.grid(True, zorder=0)
ax_country.set_title('CHURN RATE BY COUNTRY', pad=12)
ax_country.set_xlabel('Churn Rate (%)')
for bar, val in zip(hbars, country_churn.values):
    ax_country.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontsize=9,
                    fontweight='bold', color=PALETTE['text'])
 
plt.savefig('visual1_business_overview.png',
            dpi=150, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()
print(" Visual 1 saved")

VISUAL 2: Behavioral Signals


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=PALETTE['bg'])
fig.suptitle('BEHAVIORAL SIGNALS DRIVING CHURN',
             fontsize=20, fontweight='bold', color=PALETTE['text'], y=1.02)
 
behavioral_cols = ['login_frequency_per_week', 'features_used_count', 'last_login_days_ago']
col_labels      = ['Login Frequency / Week', 'Features Used (Count)', 'Days Since Last Login']
 
for ax, col, label in zip(axes, behavioral_cols, col_labels):
    churned_data  = df[df['churn'] == 1][col]
    retained_data = df[df['churn'] == 0][col]
 
    ax.hist(retained_data, bins=25, alpha=0.75, color=PALETTE['retain'],
            label='Retained', density=True, zorder=3)
    ax.hist(churned_data,  bins=25, alpha=0.75, color=PALETTE['churn'],
            label='Churned',  density=True, zorder=4)
 
    ax.axvline(churned_data.mean(),  color=PALETTE['churn'],
               linestyle='--', linewidth=1.8, alpha=0.9,
               label=f'Churn avg: {churned_data.mean():.1f}')
    ax.axvline(retained_data.mean(), color=PALETTE['retain'],
               linestyle='--', linewidth=1.8, alpha=0.9,
               label=f'Retain avg: {retained_data.mean():.1f}')
 
    ax.set_title(label, pad=10)
    ax.set_ylabel('Density')
    ax.yaxis.grid(True)
    ax.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig('visual2_behavioral_signals.png',
            dpi=150, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()
print("Visual 2 saved")

Feature Engineering & Risk Scoring

In [ ]:
def compute_risk_score(row):
    """
    Simple additive risk score (0–100).
    Each flag adds weighted points based on business logic.
    """
    score = 0
    # High support tickets
    if row['support_tickets'] > 5:
        score += 30
    # Low login frequency
    if row['login_frequency_per_week'] < 3:
        score += 20
    # Inactive recently
    if row['last_login_days_ago'] > 30:
        score += 20
    # Few features used
    if row['features_used_count'] < 5:
        score += 15
    # Monthly contract
    if row['contract_type'] == 'Monthly':
        score += 10
    # Long avg response time
    if row['avg_response_time_hours'] > 24:
        score += 10
    # Basic plan
    if row['plan_type'] == 'Basic':
        score += 5
    # Short sessions
    if row['avg_session_minutes'] < 15:
        score += 10
    return min(score, 100)
 
df['risk_score'] = df.apply(compute_risk_score, axis=1)
 
# Risk tier
def risk_tier(score):
    if score >= 65:
        return 'High Risk'
    elif score >= 35:
        return 'Medium Risk'
    else:
        return 'Low Risk'
 
df['risk_tier'] = df['risk_score'].apply(risk_tier)
 
print("Risk scoring complete")
print("\nRisk tier distribution:")
print(df['risk_tier'].value_counts())
print("\nChurn rate per risk tier:")
print(df.groupby('risk_tier')['churn'].mean().round(3) * 100)
 

VISUAL 3: Risk Score Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=PALETTE['bg'])
fig.suptitle('CUSTOMER RISK SCORE ANALYSIS',
             fontsize=20, fontweight='bold', color=PALETTE['text'], y=1.02)
 
# Left: Risk score distribution by churn
ax = axes[0]
ax.hist(df[df['churn']==0]['risk_score'], bins=30, alpha=0.75,
        color=PALETTE['retain'], label='Retained', density=True, zorder=3)
ax.hist(df[df['churn']==1]['risk_score'], bins=30, alpha=0.75,
        color=PALETTE['churn'],  label='Churned',  density=True, zorder=4)
ax.axvline(35, color=PALETTE['accent4'],  linestyle=':', linewidth=2, label='Medium threshold (35)')
ax.axvline(65, color=PALETTE['accent2'],  linestyle=':', linewidth=2, label='High threshold (65)')
ax.set_title('RISK SCORE DISTRIBUTION', pad=10)
ax.set_xlabel('Risk Score (0–100)')
ax.set_ylabel('Density')
ax.yaxis.grid(True)
ax.legend(fontsize=9)
 
# Right: Churn rate by risk tier (sorted)
ax2 = axes[1]
tier_order  = ['Low Risk', 'Medium Risk', 'High Risk']
tier_churn  = df.groupby('risk_tier')['churn'].mean() * 100
tier_counts = df['risk_tier'].value_counts()
tier_colors = [PALETTE['retain'], PALETTE['accent4'], PALETTE['churn']]
 
bars = ax2.bar(
    tier_order,
    [tier_churn.get(t, 0) for t in tier_order],
    color=tier_colors,
    edgecolor=PALETTE['bg'], linewidth=1.5, width=0.5, zorder=3
)
ax2.yaxis.grid(True, zorder=0)
ax2.set_title('CHURN RATE BY RISK TIER', pad=10)
ax2.set_ylabel('Churn Rate (%)')
 
for bar, tier in zip(bars, tier_order):
    churn_val = tier_churn.get(tier, 0)
    count_val = tier_counts.get(tier, 0)
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.8,
             f'{churn_val:.1f}%\n({count_val:,} customers)',
             ha='center', va='bottom', fontsize=10,
             fontweight='bold', color=PALETTE['text'])
 
plt.tight_layout()
plt.savefig('visual3_risk_score.png',
            dpi=150, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()
print("Visual 3 saved")

Model Training (Random Forest + Logistic Regression)

In [ ]:
# Encode categoricals
df_model = df.copy()
le        = LabelEncoder()
cat_cols  = ['gender', 'country', 'plan_type', 'contract_type', 'payment_method']
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])
 
feature_cols = [
    'gender', 'age', 'country', 'plan_type', 'contract_type',
    'monthly_fee', 'payment_method', 'login_frequency_per_week',
    'avg_session_minutes', 'features_used_count', 'last_login_days_ago',
    'support_tickets', 'avg_response_time_hours', 'risk_score'
]
 
X = df_model[feature_cols]
y = df_model['churn']
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
# ── Random Forest ─────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=150, max_depth=10,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds  = rf.predict(X_test)
rf_probs  = rf.predict_proba(X_test)[:, 1]
rf_auc    = roc_auc_score(y_test, rf_probs)
 
# ── Logistic Regression ───────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
 
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
lr_preds  = lr.predict(X_test_s)
lr_probs  = lr.predict_proba(X_test_s)[:, 1]
lr_auc    = roc_auc_score(y_test, lr_probs)
 
print("=" * 45)
print("  MODEL PERFORMANCE")
print("=" * 45)
print(f"\n🌲 Random Forest  — ROC-AUC: {rf_auc:.4f}")
print(classification_report(y_test, rf_preds, target_names=['Retained','Churned']))
print(f"\n📈 Logistic Reg.  — ROC-AUC: {lr_auc:.4f}")
print(classification_report(y_test, lr_preds, target_names=['Retained','Churned']))
 

VISUAL 4: Model Performance


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=PALETTE['bg'])
fig.suptitle('MODEL PERFORMANCE DASHBOARD',
             fontsize=20, fontweight='bold', color=PALETTE['text'], y=1.02)
 
# ── ROC Curve ────────────────────────────────────────────────
ax = axes[0]
for probs, label, color in [
    (rf_probs, f'Random Forest (AUC={rf_auc:.3f})', PALETTE['accent1']),
    (lr_probs, f'Logistic Reg  (AUC={lr_auc:.3f})', PALETTE['accent3']),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=label)
 
ax.plot([0,1],[0,1], color=PALETTE['subtext'], lw=1.5, linestyle='--', label='Random Baseline')
ax.fill_between(*roc_curve(y_test, rf_probs)[:2],
                alpha=0.10, color=PALETTE['accent1'])
ax.set_title('ROC CURVES', pad=10)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=8)
ax.yaxis.grid(True)

# ── Confusion Matrix (RF) ─────────────────────────────────────
ax2 = axes[1]
cm   = confusion_matrix(y_test, rf_preds)
cmap = LinearSegmentedColormap.from_list(
    'churn_cmap', [PALETTE['card'], PALETTE['accent1']], N=256
)
sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
            xticklabels=['Retained','Churned'],
            yticklabels=['Retained','Churned'],
            ax=ax2, linewidths=2, linecolor=PALETTE['bg'],
            annot_kws={'size': 14, 'weight': 'bold', 'color': PALETTE['text']},
            cbar_kws={'shrink': 0.8})
ax2.set_title('CONFUSION MATRIX — RANDOM FOREST', pad=10)
ax2.set_ylabel('Actual')
ax2.set_xlabel('Predicted')
ax2.tick_params(colors=PALETTE['text'])

ax3 = axes[2]
importance   = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()
colors_fi    = plt.cm.plasma(np.linspace(0.2, 0.9, len(importance)))
importance.plot(kind='barh', ax=ax3, color=colors_fi, edgecolor=PALETTE['bg'], zorder=3)
ax3.xaxis.grid(True, zorder=0)
ax3.set_title('FEATURE IMPORTANCE (RF)', pad=10)
ax3.set_xlabel('Importance Score')
 
plt.tight_layout()
output_path = 'outputs/visual4_model_performance.png'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

plt.savefig(output_path,
            dpi=150, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()
print("✅  Visual 4 saved")